# 00 — Visite guidée du dépôt

**Phase 0.** Ce notebook n'implémente rien : il lit le dépôt réel (`CLAUDE.md`, `docs/`, `packages/`, `infra/`, `.github/`) et exécute la CLI `cgp` telle qu'elle est livrée. En phase 0 les modes `sample` et `full` sont identiques : rien ne touche AWS ni Bedrock.

Contrat : `docs/09-learning-method.md` §2.

In [1]:
import os
from pathlib import Path

from cgp.repo import find_repo_root

# --- Configuration : modifie ces trois valeurs puis relance tout le notebook -------------------
RULE_TO_INSPECT = 11            # règle non négociable à détailler (1 à 14, CLAUDE.md §4)
LAYER_TO_INSPECT = "foundation"  # couche Terraform : foundation, platform, agents, frontend
WORKFLOW_TO_INSPECT = "terraform-layer"  # workflow GitHub : ci, terraform, terraform-layer, promote, drift

NB_MODE = os.environ.get("CGP_NB_MODE", "sample")
REPO = find_repo_root(Path.cwd())
print(f"mode = {NB_MODE!r}, dépôt = {REPO}")

mode = 'sample', dépôt = /Users/christophe/Sites/cgp-ai


## 1. Le problème

Un système multi-agents de conseil patrimonial mélange quatre natures de code : des règles métier (fiscalité, conformité), du calcul déterministe, des appels à un LLM, et de l'infrastructure. Lire le dépôt dans l'ordre alphabétique ne dit pas **où une règle est appliquée** ni **qui contrôle le flux**. Or c'est la seule chose qui compte pour juger le système : quand le LLM se trompe, quel code l'arrête ?

Ce notebook construit la carte « règle non négociable → emplacement dans le code », montre le trajet d'une question en 16 étapes, et exécute la seule commande livrée en phase 0, le rapport de validation.

## 2. Les options

| Option | Ce que ça donne | Pourquoi pas |
|---|---|---|
| **Un agent unique** avec tous les outils en processus | simple à démarrer | le LLM décide de l'ordre des étapes ; impossible de garantir « vérification avant conformité avant validation » |
| **Un micro-service par agent** | isolation forte | sept déploiements, sept traces à recoller, pour un seul vertical |
| **Un runtime, un pipeline en code, des Lambdas d'outils** (retenue) | l'enchaînement est du Python testable, les agents sont des étapes typées, chaque outil est isolé par IAM | plus de code d'orchestration à écrire, et c'est voulu (`CLAUDE.md` règle 11) |

## 3. Lecture guidée du dépôt réel

### 3.1 Les 14 règles non négociables et où elles vivent

Les règles sont lues dans `CLAUDE.md` §4. La table `RULE_LOCATIONS` dit où chacune est (ou sera) appliquée ; la colonne « existe » vérifie que le chemin est déjà dans le dépôt.

In [2]:
import re

from IPython.display import Markdown, display

claude_md = (REPO / "CLAUDE.md").read_text(encoding="utf-8")
section = claude_md.split("## 4. Règles non négociables")[1].split("## 5.")[0]
RULES = {int(n): title for n, title in re.findall(r"^(\d+)\. \*\*(.+?)\*\*", section, re.M)}

RULE_LOCATIONS = {
    1: ["packages/calc_engine", "packages/verification"],
    2: ["packages/params", "packages/calc_engine/tests"],
    3: ["packages/params"],
    4: ["packages/legal_rag", "packages/verification"],
    5: ["packages/domain", "services/tools_lambda", "services/profile_api"],
    6: ["services/agent_runtime", "packages/verification", "packages/compliance", "services/review_workflow"],
    7: ["infra/global/organization/scp.tf", "infra/modules/security", "infra/modules/data"],
    8: ["infra/README.md", "packages/agents"],
    9: ["infra", ".github/workflows/terraform.yml", "infra/global/organization/scp.tf"],
    10: ["docs/08-production-readiness.md", "Makefile", ".pre-commit-config.yaml", ".github/workflows/ci.yml"],
    11: ["services/agent_runtime", "packages/agents"],
    12: ["src/cgp/validation_report.py"],
    13: ["evals/runners"],
    14: ["notebooks", "docs/infra", ".github/workflows/ci.yml"],
}

rows = ["| # | Règle | Où | Existe |", "|---|---|---|---|"]
for number, title in RULES.items():
    where = ", ".join(f"`{p}`" for p in RULE_LOCATIONS[number])
    exists = all((REPO / p).exists() for p in RULE_LOCATIONS[number])
    rows.append(f"| {number} | {title} | {where} | {'oui' if exists else 'phase suivante'} |")
display(Markdown("\n".join(rows)))

| # | Règle | Où | Existe |
|---|---|---|---|
| 1 | Aucun calcul financier ou fiscal par le LLM. | `packages/calc_engine`, `packages/verification` | oui |
| 2 | Aucun paramètre réglementaire dans les prompts ni dans le code applicatif. | `packages/params`, `packages/calc_engine/tests` | oui |
| 3 | Aucune valeur réglementaire de mémoire. | `packages/params` | oui |
| 4 | Toute affirmation juridique ou fiscale d'un agent cite une source | `packages/legal_rag`, `packages/verification` | oui |
| 5 | Le profil patrimonial structuré est la source de vérité | `packages/domain`, `services/tools_lambda`, `services/profile_api` | oui |
| 6 | Aucune recommandation personnalisée n'atteint l'utilisateur sans | `services/agent_runtime`, `packages/verification`, `packages/compliance`, `services/review_workflow` | oui |
| 7 | Données personnelles | `infra/global/organization/scp.tf`, `infra/modules/security`, `infra/modules/data` | oui |
| 8 | Vérifie les API AWS avant de les utiliser. | `infra/README.md`, `packages/agents` | oui |
| 9 | Infrastructure uniquement en Terraform. | `infra`, `.github/workflows/terraform.yml`, `infra/global/organization/scp.tf` | oui |
| 10 | Qualité production dès la première ligne. | `docs/08-production-readiness.md`, `Makefile`, `.pre-commit-config.yaml`, `.github/workflows/ci.yml` | oui |
| 11 | Le contrôle du flux est du code, pas du LLM. | `services/agent_runtime`, `packages/agents` | oui |
| 12 | Tout contenu métier généré porte un statut de validation | `src/cgp/validation_report.py` | oui |
| 13 | Aucune conclusion sur une seule exécution. | `evals/runners` | oui |
| 14 | Le matériel d'apprentissage exécute le vrai code. | `notebooks`, `docs/infra`, `.github/workflows/ci.yml` | oui |

In [3]:
rule_text = re.search(rf"^{RULE_TO_INSPECT}\. (.+?)(?=^\d+\. |\Z)", section, re.M | re.S).group(1).strip()
print(f"Règle {RULE_TO_INSPECT}\n{'-' * 80}\n{rule_text}\n")
for p in RULE_LOCATIONS[RULE_TO_INSPECT]:
    target = REPO / p
    state = "existe" if target.exists() else "à venir"
    kind = "fichier" if target.is_file() else "dossier" if target.is_dir() else ""
    print(f"  {state:8} {kind:8} {p}")

Règle 11
--------------------------------------------------------------------------------
**Le contrôle du flux est du code, pas du LLM.** L'enchaînement périmètre → planification → complétude → spécialistes → vérification → conformité → validation est implémenté en Python dans le runtime ; les agents LLM sont des étapes appelées par ce code. `user_id`, `session_id`, versions, statuts et transitions ne sont jamais fournis par le LLM.

  existe   dossier  services/agent_runtime
  existe   dossier  packages/agents


### 3.2 Le trajet d'une question : 16 étapes, dont 7 agents

La table vient de `docs/03-agents.md` §2. Ce qui compte : chaque étape LLM est encadrée par du code, et le code décide de la suite (quota, périmètre, complétude, vérification, conformité, persistance).

In [4]:
agents_md = (REPO / "docs" / "03-agents.md").read_text(encoding="utf-8")
table = agents_md.split("## 2. Étapes du pipeline")[1].split("Budgets :")[0].strip()
display(Markdown(table))
steps = [line for line in table.splitlines() if re.match(r"\| \d+ \|", line)]
by_type = {}
for line in steps:
    kind = line.split("|")[3].strip()
    by_type[kind] = by_type.get(kind, 0) + 1
print({k: v for k, v in sorted(by_type.items(), key=lambda kv: -kv[1])})

| # | Étape | Type | Entrée → Sortie |
|---|---|---|---|
| 1 | Quota | code | `ToolContext` → autorisé ou réponse d'attente |
| 2 | Garde-fou d'entrée | Bedrock Guardrails | message → bloqué ou non |
| 3 | Chargement | code | profil courant, N derniers messages |
| 4 | Périmètre | code | profil + message → dans le périmètre ou redirection |
| 5 | Planification | agent `planner` | message, résumé du profil (sans montants) → `Plan` |
| 6 | Périmètre sur le plan | code | `Plan.out_of_scope_signals` → redirection éventuelle |
| 7 | Complétude | code | `Plan.domains` + profil → champs bloquants manquants |
| 8 | Collecte | agent `collector` | champs manquants → questions (≤ 3) ; ou réponses de l'utilisateur → propositions de mise à jour |
| 9 | Spécialistes | agents `tax`, `estate`, … | plan, profil, outils → `SpecialistFinding` (en parallèle) |
| 10 | Vérification déterministe | code | finding → `CheckResult[]` ; renvoi au spécialiste si échec (≤ 2) |
| 11 | Consolidation | agent `consolidator` | findings → `DraftRecommendation` |
| 12 | Vérificateur | agent `verifier` | draft, findings, profil → statut et problèmes ; renvoi au consolidateur si rejet (≤ 2) |
| 13 | Conformité | code + agent `classifier` | draft → `ComplianceDecision` |
| 14 | Réponse | code (+ agent `educator` si recommandation personnalisée) | texte final avec mentions, ou réponse d'attente |
| 15 | Garde-fou de sortie | Bedrock Guardrails | texte → bloqué ou diffusé |
| 16 | Persistance et audit | code | tout |

{'code': 7, 'Bedrock Guardrails': 2, 'agent `planner`': 1, 'agent `collector`': 1, 'agents `tax`, `estate`, …': 1, 'agent `consolidator`': 1, 'agent `verifier`': 1, 'code + agent `classifier`': 1, 'code (+ agent `educator` si recommandation personnalisée)': 1}


### 3.3 Le workspace Python : 14 distributions, un graphe de dépendances

Chaque package est une distribution `cgp-<nom>` importable sous `cgp_<nom>`. Les dépendances déclarées dessinent l'architecture : le moteur de calcul ne dépend que du domaine et des paramètres, jamais du stockage ni des agents.

In [5]:
import importlib
from importlib.metadata import distributions

dists = sorted(
    (d for d in distributions() if d.metadata["Name"].startswith("cgp")),
    key=lambda d: d.metadata["Name"],
)
rows = ["| Distribution | Module | Version | Dépend de |", "|---|---|---|---|"]
for d in dists:
    name = d.metadata["Name"]
    module = name.replace("-", "_")
    version = importlib.import_module(module).__version__
    requires = [r.split(" ")[0] for r in (d.requires or []) if r.startswith("cgp-")]
    rows.append(f"| `{name}` | `{module}` | {version} | {', '.join(requires) or '—'} |")
display(Markdown("\n".join(rows)))

| Distribution | Module | Version | Dépend de |
|---|---|---|---|
| `cgp` | `cgp` | 0.1.0 | cgp-domain, cgp-calc-engine, cgp-params, cgp-storage, cgp-legal-rag, cgp-compliance, cgp-verification, cgp-agents, cgp-agent-runtime, cgp-tools-lambda, cgp-profile-api, cgp-review-workflow, cgp-ingestion, cgp-migrations |
| `cgp-agent-runtime` | `cgp_agent_runtime` | 0.1.0 | cgp-agents, cgp-domain, cgp-compliance, cgp-verification |
| `cgp-agents` | `cgp_agents` | 0.1.0 | cgp-domain, cgp-calc-engine, cgp-params, cgp-compliance, cgp-verification, cgp-legal-rag |
| `cgp-calc-engine` | `cgp_calc_engine` | 0.1.0 | cgp-domain, cgp-params |
| `cgp-compliance` | `cgp_compliance` | 0.1.0 | cgp-domain, cgp-params |
| `cgp-domain` | `cgp_domain` | 0.1.0 | — |
| `cgp-ingestion` | `cgp_ingestion` | 0.1.0 | cgp-legal-rag, cgp-storage |
| `cgp-legal-rag` | `cgp_legal_rag` | 0.1.0 | cgp-storage |
| `cgp-migrations` | `cgp_migrations` | 0.1.0 | cgp-storage |
| `cgp-params` | `cgp_params` | 0.1.0 | — |
| `cgp-profile-api` | `cgp_profile_api` | 0.1.0 | cgp-domain, cgp-storage |
| `cgp-review-workflow` | `cgp_review_workflow` | 0.1.0 | cgp-storage |
| `cgp-storage` | `cgp_storage` | 0.1.0 | cgp-domain |
| `cgp-tools-lambda` | `cgp_tools_lambda` | 0.1.0 | cgp-domain, cgp-calc-engine, cgp-params, cgp-legal-rag, cgp-storage |
| `cgp-verification` | `cgp_verification` | 0.1.0 | cgp-domain, cgp-calc-engine, cgp-legal-rag |

### 3.4 Le code livré en phase 0 : le rapport de validation

`cgp validation-report` est la première brique de la règle 12 : tout contenu métier porte un statut et apparaît dans le rapport. En phase 0, deux collecteurs ; les suivants s'ajoutent avec les contenus qu'ils inventorient. Le code est affiché depuis le module réel.

In [6]:
import inspect

from cgp import validation_report

print(inspect.getsource(validation_report.Collector))
print(inspect.getsource(validation_report.DocsMarkerCollector))

class Collector(Protocol):
    """Un collecteur inventorie une famille de contenu du dépôt."""

    name: str

    def collect(self, repo_root: Path) -> list[PendingItem]:
        """Renvoie les éléments en attente trouvés sous `repo_root`."""
        ...

class DocsMarkerCollector:
    """Relève chaque ligne de `docs/` portant le marqueur **[À VALIDER]**."""

    name = "docs"
    docs_dir = Path("docs")

    def collect(self, repo_root: Path) -> list[PendingItem]:
        """Parcourt `docs/**/*.md` et relève les lignes marquées."""
        items: list[PendingItem] = []
        for path in sorted((repo_root / self.docs_dir).rglob("*.md")):
            lines = path.read_text(encoding="utf-8").splitlines()
            for lineno, line in enumerate(lines, start=1):
                if VALIDATION_MARKER in line:
                    items.append(
                        PendingItem(
                            category=self.name,
                            location=f"{path.relative_to(repo

### 3.5 L'infrastructure : 3 comptes × 4 couches, 11 modules

Chaque racine `infra/envs/<env>/<couche>/main.tf` appelle ses modules. Les SCP de `infra/global/organization/scp.tf` bornent les comptes membres.

In [7]:
envs = ["dev", "staging", "prod"]
layers = ["foundation", "platform", "agents", "frontend"]
matrix = {}
for env in envs:
    for layer in layers:
        main_tf = (REPO / "infra" / "envs" / env / layer / "main.tf").read_text(encoding="utf-8")
        matrix[(env, layer)] = re.findall(r'^module "(\w+)"', main_tf, re.M)
rows = ["| Couche | Modules | Identique sur les 3 comptes |", "|---|---|---|"]
for layer in layers:
    mods = matrix[("dev", layer)]
    same = all(matrix[(env, layer)] == mods for env in envs)
    rows.append(f"| `{layer}` | {', '.join(f'`{m}`' for m in mods)} | {'oui' if same else 'NON'} |")
display(Markdown("\n".join(rows)))

print(f"\nCouche {LAYER_TO_INSPECT!r} : README du premier module")
first = matrix[("dev", LAYER_TO_INSPECT)][0]
print((REPO / "infra" / "modules" / first / "README.md").read_text(encoding="utf-8")[:600])

scp = (REPO / "infra" / "global" / "organization" / "scp.tf").read_text(encoding="utf-8")
print("\nSCP (statements) :", re.findall(r'sid\s+= "(\w+)"', scp))

| Couche | Modules | Identique sur les 3 comptes |
|---|---|---|
| `foundation` | `network`, `security`, `data` | oui |
| `platform` | `tools`, `api`, `gateway`, `review`, `ingestion`, `observability` | oui |
| `agents` | `agents` | oui |
| `frontend` | `frontend` | oui |


Couche 'foundation' : README du premier module
# Module `network`

VPC privé sans IP publique, sous-réseaux privés, endpoints d'interface (Bedrock Runtime, Secrets Manager, CloudWatch Logs, Lambda, STS) et de passerelle (S3), groupes de sécurité.

- **Couche** : `foundation` (`docs/01-architecture.md` §3).
- **Phase de livraison** : 3 (`docs/07-roadmap.md`).
- **Guide de lecture** : `docs/infra/01-network.md`, écrit avec le module.

## Entrées

| Variable | Description |
|---|---|
| `project` | Nom court du projet, préfixe des ressources. |
| `env` | Environnement cible (`dev`, `staging`, `prod`). |
| `tags` | Tags communs ; `project`, `en

SCP (statements) : ['DenyOutsideEuropeanUnion', 'DenyIamUsersAndAccessKeys', 'DenyLeavingOrganization', 'ProtectCloudTrail', 'ProtectKmsKeys', 'ProtectBackups']


### 3.6 La CI : cinq workflows, un seul chemin vers `prod`

Les workflows sont lus tels quels. Remarque de lecture YAML : la clé `on` est interprétée comme le booléen `True` par PyYAML, d'où le `wf.get(True)`.

In [8]:
import yaml

workflows = {}
for path in sorted((REPO / ".github" / "workflows").glob("*.yml")):
    wf = yaml.safe_load(path.read_text(encoding="utf-8"))
    triggers = wf.get("on", wf.get(True))
    triggers = list(triggers) if isinstance(triggers, dict) else [triggers]
    workflows[path.stem] = {"name": wf["name"], "triggers": triggers, "jobs": wf["jobs"]}
    print(f"{path.name:22} déclencheurs={triggers}  jobs={list(wf['jobs'])}")

print(f"\nGraphe du workflow {WORKFLOW_TO_INSPECT!r} :")
for job, spec in workflows[WORKFLOW_TO_INSPECT]["jobs"].items():
    needs = spec.get("needs", [])
    needs = [needs] if isinstance(needs, str) else needs
    env = spec.get("environment")
    print(f"  {job:24} needs={needs}  environment={env}")

ci.yml                 déclencheurs=['pull_request', 'push']  jobs=['python', 'notebooks', 'audit', 'secrets']
drift.yml              déclencheurs=['schedule', 'workflow_dispatch']  jobs=['plan']
promote.yml            déclencheurs=['workflow_dispatch']  jobs=['guard', 'staging', 'prod']
terraform-layer.yml    déclencheurs=['workflow_call']  jobs=['plan', 'apply']
terraform.yml          déclencheurs=['pull_request', 'push']  jobs=['static', 'plan-dev', 'comment', 'apply-dev-foundation', 'apply-dev-platform', 'apply-dev-agents', 'apply-dev-frontend']

Graphe du workflow 'terraform-layer' :
  plan                     needs=[]  environment=None
  apply                    needs=['plan']  environment=${{ inputs.env }}


## 4. Exécution : le rapport de validation, en vrai

Le rapport s'écrit dans un répertoire temporaire pour ne pas polluer le dépôt. On regarde ce qu'il contient : en phase 0, uniquement les points **[À VALIDER]** de `docs/` (aucun prompt, aucun paramètre encore).

In [9]:
import tempfile
from collections import Counter

from cgp.validation_report import write_report

with tempfile.TemporaryDirectory() as tmp:
    target, items = write_report(REPO, Path(tmp) / "validation.md")
    report = target.read_text(encoding="utf-8")

print(Counter(item.category for item in items))
print("\n".join(report.splitlines()[:16]))
print("…")

Counter({'docs': 12})
# Rapport de validation

Généré le 2026-09-18T12:59:53+00:00.

Aucune validation professionnelle n'est prévue dans ce projet : tout contenu métier reste `draft`. Ce rapport liste ce qu'une relecture devrait couvrir.

**Total : 12 élément(s) en attente.**

## docs (12)

| Emplacement | Statut | Résumé |
|---|---|---|
| `docs/02-domain-model.md:208` | à valider | sustainability_preferences: Literal["none", "some", "strong"] | None = None # granularité attendue par le questionnaire… |
| `docs/02-domain-model.md:263` | à valider | patrimoine brut supérieur au seuil `scope.max_gross_wealth` du référentiel de paramètres (** **, valeur initiale 5 000 … |
| `docs/03-agents.md:124` | à valider | Règles déterministes (`packages/compliance/rules/classification.yaml`) : une sortie est `recommandation_personnalisee` … |
| `docs/05-safety-compliance.md:3` | à valider | Ce document décrit des exigences techniques. Le cadre réglementaire (statut de conseiller en investissements f

## 5. Les pièges (phase 0)

- **`ruff format` a reformaté les blocs Python de `docs/02` et `docs/04`** : le formateur traite les fences Markdown. Les docs sont la spécification ; elles sont désormais exclues de ruff et ont été restaurées à l'identique.
- **`uv sync` n'installe pas les membres du workspace** que la racine ne déclare pas : les tests d'import échouaient avec `ModuleNotFoundError`. La CLI dépend maintenant explicitement des 14 packages.
- **zsh ne découpe pas les variables non citées** dans `for x in $liste` : un générateur de `pyproject.toml` a produit une dépendance unique contenant six noms.
- **Un backend S3 partiel exige `init`** même pour un plan : le bootstrap passe par `backend_override.tf` (backend local), puis migration.
- **Le compte de gestion échappe aux SCP** : l'utilisateur IAM administrateur y était sans MFA avec des clés statiques (guide 00, étape 1).

## 6. Expérimentations guidées

Dans la cellule de configuration en tête :

1. `RULE_TO_INSPECT = 7`, puis `= 13` : compare une règle déjà outillée (SCP régions UE) et une règle qui attend la phase 5 (N exécutions, intervalles).
2. `LAYER_TO_INSPECT = "platform"` : la couche la plus peuplée, six modules, tous vides pour l'instant.
3. `WORKFLOW_TO_INSPECT = "terraform"` : observe la chaîne `apply-dev-foundation → platform → agents → frontend` et le job `comment`.

## 7. Les chiffres

Phase 0 : aucun aléa (pas de LLM, pas de réseau), donc pas d'intervalle de confiance. Les chiffres sont des comptages de structure ; ils servent de point de départ pour mesurer la croissance du dépôt.

In [10]:
import json

counts = {
    "packages": len(list((REPO / "packages").iterdir())),
    "services": len(list((REPO / "services").iterdir())),
    "modules_terraform": len([p for p in (REPO / "infra" / "modules").iterdir() if p.is_dir()]),
    "racines_terraform": len(list((REPO / "infra" / "envs").glob("*/*/main.tf")))
    + len(list((REPO / "infra" / "global").glob("**/main.tf"))) - len(list((REPO / "infra" / "global" / "modules").glob("*/main.tf"))),
    "workflows": len(workflows),
    "regles_non_negociables": len(RULES),
    "points_a_valider": sum(1 for item in items if item.category == "docs"),
    "guides_infra": len(list((REPO / "docs" / "infra").glob("[0-9][0-9]-*.md"))),
    "etapes_pipeline": len(steps),
}
for key, value in counts.items():
    print(f"{key:24} {value}")

results_dir = os.environ.get("CGP_NB_RESULTS_DIR")
if results_dir:
    out = REPO / results_dir
    out.mkdir(parents=True, exist_ok=True)
    (out / "00_visite_guidee.json").write_text(json.dumps(counts, indent=2), encoding="utf-8")
    print(f"\narchivé dans {out / '00_visite_guidee.json'}")

packages                 8
services                 6
modules_terraform        11
racines_terraform        16
workflows                5
regles_non_negociables   14
points_a_valider         12
guides_infra             2
etapes_pipeline          16


## 8. Questions d'entretien

<details><summary>1. Pourquoi le contrôle du flux est-il du code et non un agent orchestrateur ?</summary>

Parce qu'un ordre d'étapes imposé par la loi ou la conformité (vérification avant diffusion, validation humaine avant recommandation) ne peut pas dépendre d'une décision probabiliste. Le code garantit l'ordre, les budgets et les itérations ; le LLM produit des sorties typées à l'intérieur de chaque étape.
</details>

<details><summary>2. Comment empêche-t-on un agent de lire le profil d'un autre utilisateur ?</summary>

`user_id` vient du JWT validé par le runtime, est placé dans un `ToolContext` construit par le code, et n'apparaît dans aucun schéma exposé au LLM. Les Lambdas d'outils n'acceptent que des rôles IAM autorisés et filtrent toute requête par ce contexte. Un test d'accès croisé le vérifie.
</details>

<details><summary>3. Que gagne-t-on à découper l'état Terraform par environnement et par couche ?</summary>

Un rayon d'impact borné : un `apply` de la couche `agents` ne peut pas toucher la base de données, et une erreur sur `dev` ne peut pas atteindre `prod` (comptes distincts, `allowed_account_ids`). Le prix : des `terraform_remote_state` entre couches et quatre plans par environnement.
</details>

<details><summary>4. Pourquoi la CI applique-t-elle un artefact de plan plutôt que de replanifier ?</summary>

Pour que ce qui est appliqué en `prod` soit exactement ce que le relecteur a lu. Terraform refuse un plan dont l'état a bougé entre-temps. Sur un dépôt public, l'artefact est chiffré parce qu'il contient l'état.
</details>

<details><summary>5. Qu'est-ce que le rapport de validation prouve, puisque personne ne valide ?</summary>

Qu'aucun contenu métier n'est présenté comme validé : chaque paramètre, cas, règle ou prompt porte `draft` et la source consultée. Il montre où une relecture professionnelle se brancherait, et le mode démonstration l'affiche à l'utilisateur.
</details>

## 9. Ce qu'on n'a pas fait, et pourquoi

- **Aucun appel AWS dans ce notebook**, même en mode `full` : la phase 0 ne déploie rien d'applicatif ; les vérifications AWS sont des commandes CLI dans `docs/infra/00-bootstrap.md` §4.
- **Pas de schéma d'architecture dessiné ici** : `docs/01-architecture.md` §1 le porte ; le notebook préfère lire les fichiers qui font foi.
- **Pas de mesure** : rien n'est aléatoire en phase 0. Les premiers intervalles de confiance arrivent avec l'ablation du RAG (notebook 03).